# Capítulo 6: Transformers y Large Language Models (LLMs)

## Objetivos de aprendizaje

- Comprender la arquitectura Transformer y el mecanismo de atención.
- Entender el concepto de transferencia de aprendizaje y fine-tuning.
- Conocer los principales modelos pre-entrenados (BERT, GPT).
- Utilizar LLMs para tareas de generación, resumen y modificación de texto.

## 6.1 De Word2Vec a Transformers

### Limitaciones de Word2Vec

- **Representación estática**: "banco" tiene el mismo vector sin importar si es un asiento o una entidad financiera.
- **Sin contexto**: No captura el significado según el uso en la oración.

### La evolución

```
BoW/TF-IDF → Word2Vec/GloVe → ELMo → Transformer → BERT/GPT → LLMs
(disperso)    (denso estático)  (contextual)  (atención)  (pre-entrenado)  (masivo)
```

Los **Transformers** (Vaswani et al., 2017) introducen representaciones **contextuales**: cada palabra obtiene un vector diferente según su contexto.

## 6.2 La arquitectura Transformer

### Componentes clave

1. **Self-Attention (Auto-atención)**: Permite a cada token "atender" a todos los demás tokens de la secuencia, capturando dependencias a cualquier distancia.

2. **Multi-Head Attention**: Múltiples cabezas de atención en paralelo capturan diferentes tipos de relaciones.

3. **Positional Encoding**: Inyecta información sobre la posición de cada token en la secuencia.

4. **Feed-Forward Network**: Red neuronal que procesa cada posición independientemente.

### Fórmula de atención

$$\text{Attention}(Q, K, V) = \text{softmax}\left(\frac{QK^T}{\sqrt{d_k}}\right)V$$

Donde:
- $Q$ (Query), $K$ (Key), $V$ (Value) son proyecciones lineales de la entrada.
- $d_k$ es la dimensión de las claves (factor de escala).

### Encoder vs. Decoder

| Componente | Modelo | Tarea principal |
|-----------|--------|-----------------|
| **Encoder** | BERT, RoBERTa | Comprensión (clasificación, NER, QA) |
| **Decoder** | GPT, LLaMA | Generación de texto |
| **Encoder-Decoder** | T5, BART | Traducción, resumen |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Implementación simplificada de Self-Attention
def self_attention(X, d_k=None):
    """Calcula self-attention simplificada (sin proyecciones aprendidas)."""
    if d_k is None:
        d_k = X.shape[1]
    # Q, K, V son la misma entrada (self-attention simplificada)
    scores = X @ X.T / np.sqrt(d_k)  # (n x n)
    weights = np.exp(scores) / np.exp(scores).sum(axis=1, keepdims=True)  # softmax
    output = weights @ X  # (n x d)
    return output, weights

# Ejemplo: 4 tokens con embeddings de dimensión 3
np.random.seed(42)
tokens = ["el", "gato", "persigue", "ratón"]
X = np.random.randn(4, 3)

output, weights = self_attention(X)

# Visualizar pesos de atención
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(weights, cmap='Blues')
ax.set_xticks(range(len(tokens)))
ax.set_yticks(range(len(tokens)))
ax.set_xticklabels(tokens, fontsize=12)
ax.set_yticklabels(tokens, fontsize=12)
ax.set_title('Pesos de Self-Attention', fontsize=14)

# Agregar valores
for i in range(len(tokens)):
    for j in range(len(tokens)):
        ax.text(j, i, f'{weights[i,j]:.2f}', ha='center', va='center', fontsize=11)

plt.colorbar(im)
plt.tight_layout()
plt.show()

## 6.3 BERT: Bidirectional Encoder Representations from Transformers

**BERT** (Devlin et al., 2019) es un modelo Transformer pre-entrenado bidireccionalmente. Esto significa que para representar una palabra, considera tanto el contexto izquierdo como derecho simultáneamente.

### Pre-entrenamiento

BERT se pre-entrena con dos tareas:
1. **Masked Language Model (MLM)**: Se enmascara el 15% de los tokens y el modelo aprende a predecirlos.
2. **Next Sentence Prediction (NSP)**: El modelo predice si dos oraciones son consecutivas.

### Fine-tuning

Después del pre-entrenamiento, BERT se puede ajustar (*fine-tune*) para tareas específicas con relativamente pocos datos etiquetados.

In [ ]:
from transformers import pipeline

# Ejemplo 1: Fill-mask (tarea MLM de BERT)
fill_mask = pipeline('fill-mask', model='dccuchile/bert-base-spanish-wwm-uncased')

resultado = fill_mask('La inteligencia [MASK] está transformando la industria.')

print("Fill-mask: 'La inteligencia [MASK] está transformando la industria.'\n")
for r in resultado[:3]:
    print(f"  {r['token_str']:15s} (score: {r['score']:.4f})")

In [ ]:
# Ejemplo 2: Análisis de sentimiento con modelo pre-entrenado
clasificador = pipeline(
    'sentiment-analysis',
    model='nlptown/bert-base-multilingual-uncased-sentiment'
)

textos = [
    "Este producto es excelente, me encantó",
    "Pésima calidad, no lo recomiendo",
    "Es un producto aceptable, nada especial"
]

print("Análisis de sentimiento:\n")
for texto in textos:
    resultado = clasificador(texto)[0]
    print(f"  '{texto}'")
    print(f"    → {resultado['label']} (confianza: {resultado['score']:.3f})\n")

## 6.4 Transferencia de aprendizaje y Fine-tuning

La **transferencia de aprendizaje** permite usar conocimiento aprendido en una tarea general (pre-entrenamiento en grandes corpus) y adaptarlo a una tarea específica.

### Proceso

```
1. Pre-entrenamiento (corpus masivo, tarea general)
   → Modelo aprende representaciones generales del lenguaje

2. Fine-tuning (dataset específico, tarea objetivo)
   → Se ajustan los pesos del modelo para la tarea concreta
```

### Ventajas

- Requiere muchos menos datos etiquetados.
- Converge más rápido.
- Obtiene mejores resultados que entrenar desde cero.

In [ ]:
# Ejemplo de extracción de embeddings contextuales con BERT
from transformers import AutoTokenizer, AutoModel
import torch

tokenizer = AutoTokenizer.from_pretrained('dccuchile/bert-base-spanish-wwm-uncased')
model = AutoModel.from_pretrained('dccuchile/bert-base-spanish-wwm-uncased')

# Obtener embeddings contextuales
texto = "El banco ofrece préstamos a bajo interés"
inputs = tokenizer(texto, return_tensors='pt', padding=True, truncation=True)

with torch.no_grad():
    outputs = model(**inputs)

# outputs.last_hidden_state: embeddings contextuales de cada token
embeddings = outputs.last_hidden_state
print(f"Texto: '{texto}'")
print(f"Tokens: {tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])}")
print(f"Shape de embeddings: {embeddings.shape}")
print(f"  → {embeddings.shape[1]} tokens x {embeddings.shape[2]} dimensiones")

## 6.5 Large Language Models (LLMs)

Los **LLMs** son modelos de lenguaje masivos (billones de parámetros) entrenados en enormes corpus de texto. Representan el estado del arte en NLP.

### Características

- **Escala**: Desde millones hasta billones de parámetros.
- **Capacidades emergentes**: A mayor escala, surgen capacidades nuevas (razonamiento, planificación).
- **Few-shot / Zero-shot**: Pueden realizar tareas con pocos o ningún ejemplo.
- **Instrucciones**: Modelos ajustados para seguir instrucciones humanas.

### Modelos destacados

| Modelo | Organización | Parámetros | Tipo |
|--------|-------------|------------|------|
| GPT-4 | OpenAI | ~1.8T (estimado) | Cerrado |
| Claude | Anthropic | No publicado | Cerrado |
| LLaMA 3 | Meta | 8B - 405B | Abierto |
| Gemini | Google | No publicado | Cerrado |
| Mistral | Mistral AI | 7B - 8x22B | Abierto |

In [ ]:
# Ejemplo: Generación de texto con un modelo generativo
generador = pipeline(
    'text-generation',
    model='datificate/gpt2-small-spanish',
    max_new_tokens=50
)

prompt = "La inteligencia artificial en la medicina permite"
resultado = generador(prompt, num_return_sequences=1)

print("Generación de texto:")
print(f"  Prompt: '{prompt}'")
print(f"  Generado: '{resultado[0]['generated_text']}'")

In [ ]:
# Ejemplo: Resumen de texto
resumidor = pipeline(
    'summarization',
    model='facebook/bart-large-cnn'
)

texto_largo = """
Artificial intelligence has transformed the way businesses operate across
multiple industries. In healthcare, AI systems can analyze medical images
to detect diseases earlier than human doctors. In finance, machine learning
algorithms process vast amounts of market data to make trading decisions
in milliseconds. The legal industry uses natural language processing to
review thousands of documents during discovery processes. Customer service
departments deploy chatbots that can handle routine inquiries, freeing
human agents for more complex issues. Despite these advances, challenges
remain in ensuring AI systems are fair, transparent, and accountable.
"""

resumen = resumidor(texto_largo, max_length=60, min_length=20)
print("Resumen:")
print(f"  {resumen[0]['summary_text']}")

## 6.6 Prompting: El arte de comunicarse con LLMs

El **prompting** es la técnica de diseñar instrucciones efectivas para obtener la respuesta deseada de un LLM.

### Tipos de prompting

| Técnica | Descripción | Ejemplo |
|---------|-------------|---------|
| **Zero-shot** | Sin ejemplos | "Clasifica el sentimiento: ..." |
| **Few-shot** | Con ejemplos | "Positivo: me encanta. Negativo: lo odio. Clasifica: ..." |
| **Chain-of-thought** | Razonamiento paso a paso | "Piensa paso a paso..." |
| **Instrucciones** | Instrucciones explícitas | "Eres un experto en..." |

## Resumen

En este capítulo aprendimos:

- **Transformers**: Arquitectura basada en atención que genera representaciones contextuales.
- **Self-Attention**: Mecanismo que permite a cada token atender a todos los demás.
- **BERT**: Modelo encoder pre-entrenado bidireccionalmente, ideal para comprensión.
- **Fine-tuning**: Adaptación de modelos pre-entrenados a tareas específicas.
- **LLMs**: Modelos masivos con capacidades de generación, resumen y razonamiento.
- **Prompting**: Técnicas para comunicarse efectivamente con LLMs.

En el próximo capítulo exploraremos cómo integrar LLMs con herramientas externas para crear **Agentes de IA**.